In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

In [ ]:
# Load individual CSVs (run the individual scraper notebooks first)
tricon = pd.read_csv('tricon.csv')
progress = pd.read_csv('progress_residential.csv')
invh = pd.read_csv('invitation_homes.csv')
amh = pd.read_csv('amh.csv')
msr = pd.read_csv('main_street_renewal.csv')
fkh = pd.read_csv('firstkey_homes.csv')

print(f'Tricon:               {len(tricon):>5} rows')
print(f'Progress Residential: {len(progress):>5} rows')
print(f'Invitation Homes:     {len(invh):>5} rows')
print(f'AMH:                  {len(amh):>5} rows')
print(f'Main Street Renewal:  {len(msr):>5} rows')
print(f'FirstKey Homes:       {len(fkh):>5} rows')

In [ ]:
# Normalize each dataset to a common schema
COMMON_COLS = [
    'source', 'street', 'city', 'state', 'zip', 'beds', 'baths',
    'sqft', 'price', 'status', 'date_available', 'special',
    'lat', 'lng', 'link', 'metro_location', 'scraped_at',
]

now = datetime.now().strftime('%Y-%m-%d %H:%M')


def normalize_tricon(df):
    return pd.DataFrame({
        'source': 'Tricon Residential',
        'street': df['address'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip_code'],
        'beds': df['bed'],
        'baths': df['bath'],
        'sqft': df['square_feet'],
        'price': df['price'],
        'status': df['available'],
        'date_available': '',
        'special': df['special'],
        'lat': None,
        'lng': None,
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


def normalize_progress(df):
    return pd.DataFrame({
        'source': 'Progress Residential',
        'street': df['street'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip'],
        'beds': df['beds'],
        'baths': df['baths'],
        'sqft': df['sqft'],
        'price': df['current_price'],
        'status': df['property_status'],
        'date_available': df['date_available'],
        'special': df['banner_status'],
        'lat': df['lat'],
        'lng': df['lng'],
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


def normalize_invh(df):
    return pd.DataFrame({
        'source': 'Invitation Homes',
        'street': df['street'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip'],
        'beds': df['beds'],
        'baths': df['baths'],
        'sqft': df['sqft'],
        'price': df['rent'],
        'status': df['status'],
        'date_available': df['available_on'],
        'special': df['is_on_special'].map({True: 'Special Available', False: ''}),
        'lat': df['lat'],
        'lng': df['lng'],
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


def normalize_amh(df):
    return pd.DataFrame({
        'source': 'AMH',
        'street': df['street'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip'],
        'beds': df['beds'],
        'baths': df['baths'],
        'sqft': df['sqft'],
        'price': df['rent'],
        'status': '',
        'date_available': df['available_date'],
        'special': '',
        'lat': df['lat'],
        'lng': df['lng'],
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


def normalize_msr(df):
    return pd.DataFrame({
        'source': 'Main Street Renewal',
        'street': df['street'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip'],
        'beds': df['beds'],
        'baths': df['baths'],
        'sqft': df['sqft'],
        'price': df['rent'],
        'status': df['listing_status'],
        'date_available': df['available_date'],
        'special': df['special'],
        'lat': df['lat'],
        'lng': df['lng'],
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


def normalize_fkh(df):
    return pd.DataFrame({
        'source': 'FirstKey Homes',
        'street': df['address'],
        'city': df['city'],
        'state': df['state'],
        'zip': df['zip'],
        'beds': df['bedrooms'],
        'baths': df['bathrooms'],
        'sqft': df['sqft'],
        'price': df['rent'],
        'status': df['special_offer'].map({True: 'Special Offer', False: ''}),
        'date_available': df['available_at'],
        'special': df['special_offer'].map({True: 'Special Offer', False: ''}),
        'lat': df['lat'],
        'lng': df['lng'],
        'link': df['link'],
        'metro_location': df['metro_location'],
        'scraped_at': now,
    })


t = normalize_tricon(tricon)
p = normalize_progress(progress)
i = normalize_invh(invh)
a = normalize_amh(amh)
m = normalize_msr(msr)
f = normalize_fkh(fkh)

print(f'Tricon normalized:    {len(t)} rows, {len(t.columns)} cols')
print(f'Progress normalized:  {len(p)} rows, {len(p.columns)} cols')
print(f'INVH normalized:      {len(i)} rows, {len(i.columns)} cols')
print(f'AMH normalized:       {len(a)} rows, {len(a.columns)} cols')
print(f'MSR normalized:       {len(m)} rows, {len(m.columns)} cols')
print(f'FKH normalized:       {len(f)} rows, {len(f.columns)} cols')

In [ ]:
combined = pd.concat([t, p, i, a, m, f], ignore_index=True)

combined['price'] = pd.to_numeric(combined['price'], errors='coerce')
combined['beds'] = pd.to_numeric(combined['beds'], errors='coerce')
combined['baths'] = pd.to_numeric(combined['baths'], errors='coerce')
combined['sqft'] = pd.to_numeric(combined['sqft'], errors='coerce')

print(f'Combined dataset: {len(combined)} rows')
print()
print(combined['source'].value_counts())
print()
combined.info()

In [5]:
summary = combined.groupby('source').agg(
    count=('price', 'size'),
    avg_price=('price', 'mean'),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    avg_sqft=('sqft', 'mean'),
    avg_beds=('beds', 'mean'),
).round(0)

summary

,count,avg_price,min_price,max_price,avg_sqft,avg_beds
source,,,,,,
AMH,117,2153.0,1595.0,3595.0,2081.0,4.0
Invitation Homes,140,2014.0,1495.0,2849.0,2012.0,4.0
Main Street Renewal,61,1944.0,1520.0,2670.0,1799.0,4.0
Progress Residential,225,1990.0,1395.0,3305.0,1915.0,3.0
Tricon Residential,173,1990.0,1539.0,3099.0,1679.0,3.0


In [6]:
combined.to_csv('combined_listings.csv', index=False)
print(f'Saved {len(combined)} rows to combined_listings.csv')

Saved 716 rows to combined_listings.csv


In [7]:
conn = sqlite3.connect('combined_listings.db')
combined.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(combined)} rows to combined_listings.db -> listings table')
conn.close()

Wrote 716 rows to combined_listings.db -> listings table
